In [15]:
import numpy as np

from line_solver import *

model = Network('model')

node = np.empty(4, dtype=object)
node[0] = Delay(model, 'Delay')
node[1] = Queue(model, 'Queue1', SchedStrategy.FCFS)
node[2] = Source(model,'Source')
node[3] = Sink(model,'Sink')

jobclass = OpenClass(model, 'Class1', 0)

node[0].setService(jobclass, HyperExp(0.5,3.0,10.0))
node[1].setService(jobclass, Exp(1))
node[2].setArrival(jobclass, Exp(0.1))

M = model.getNumberOfStations()
K = model.getNumberOfClasses()

P = model.initRoutingMatrix()
pmatrix = [[0,1,0,0],[0,0,0,1],[1,0,0,0],[0,0,0,0]]
for i in range(len(node)):
    for j in range(len(node)):
        P.set(jobclass, jobclass, node[i], node[j], pmatrix[i][j])

model.link(P)

This example shows the execution of the solver on a 1-class 2-node open model.

In [16]:
#options = Solver.defaultOptions
#options.keep = True
#options.verbose = 1
#options.cutoff = 10
#options.seed = 23000
#options.iter_max = 200
#options.samples = 2000

# This part illustrates the execution of different solvers
# TODO: SSA result is buggy 
solver = np.empty(5, dtype=object)
solver[0] = SolverCTMC(model,'keep',True)
solver[1] = SolverJMT(model,'seed',23000,'verbose',False,'keep',True)
solver[2] = SolverSSA(model,'seed',23000,'verbose',True,'samples',50000)
solver[3] = SolverFluid(model)
solver[4] = SolverMVA(model)
#solver[5] = SolverNC(model,'exact')
#solver[6] = SolverMAM(model)
#solver[7] = LINE(model)

AvgTable = np.empty(len(solver), dtype=object)
ran = list(range(len(solver)))
ran.remove(0)
for s in ran:
    print(f'\nSOLVER: {solver[s].getName()}')
    AvgTable[s] = solver[s].getAvgTable()
    print(AvgTable[s])


SOLVER: SolverJMT
  Station JobClass      QLen      Util     RespT    ResidT      Tput
0   Delay   Class1  0.022679  0.022679  0.214832  0.214832  0.099607
1  Queue1   Class1  0.113320  0.102224  1.107529  1.107529  0.100012
2  Source   Class1  0.000000  0.000000  0.000000  0.000000  0.099621

SOLVER: SolverSSA
  Station JobClass  QLen  Util  RespT  ResidT     Tput
0   Delay   Class1   0.0   0.0    0.0     0.0  0.00000
1  Queue1   Class1   0.0   0.0    0.0     0.0  0.00000
2  Source   Class1   0.0   0.0    0.0     0.0  0.10251

SOLVER: SolverFluid
  Station JobClass      QLen      Util     RespT    ResidT  Tput
0   Delay   Class1  0.021667  0.021667  0.216667  0.216667   0.1
1  Queue1   Class1  0.100000  0.100000  1.000000  1.000000   0.1
2  Source   Class1  0.000000  0.000000  0.000000  0.000000   0.1

SOLVER: SolverMVA
  Station JobClass      QLen      Util     RespT    ResidT  Tput
0   Delay   Class1  0.021665  0.021667  0.216653  0.216653   0.1
1  Queue1   Class1  0.111085  0.1000